In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Configuration for October 2026 passenger journey data

CSV_PATH = "/Volumes/workspace/default/dublin_transport_gtfs/passenger_journeys.csv"

BRONZE_TABLE = "workspace.default.passenger_journeys"
SILVER_TABLE = "workspace.default.passenger_journeys_silver"

EXPECTED_START = "2026-09-30"
EXPECTED_END = "2026-10-30"
EXPECTED_ROWS = 1000

assert "YOUR_VOLUME" not in CSV_PATH, "Set CSV_PATH to your uploaded CSV's actual Volume path."

print("CSV source:", CSV_PATH)

CSV source: /Volumes/workspace/default/dublin_transport_gtfs/passenger_journeys.csv


In [0]:
# Read as strings first: IDs are identifiers, not numeric measures.
raw_csv = (spark.read.option("header", True).option("inferSchema", False)
           .option("mode", "FAILFAST").csv(CSV_PATH))
required = ["journey_id", "journey_date", "day_of_week", "trip_id", "route_id",
            "transport_mode", "origin_stop_id", "origin_stop_name", "origin_stop_sequence",
            "destination_stop_id", "destination_stop_name", "destination_stop_sequence",
            "scheduled_departure_time", "scheduled_arrival_time",
            "scheduled_duration_minutes", "passenger_category", "payment_method",
            "fare_amount", "validation_status", "compliance_status", "is_weekend", "route_type"]
missing = sorted(set(required) - set(raw_csv.columns))
assert not missing, f"Missing columns: {missing}"
assert len(raw_csv.columns) == len(required), "Unexpected CSV columns: inspect source before proceeding."

passenger_journeys = (raw_csv
    .withColumn("journey_date", F.to_date("journey_date", "yyyy-MM-dd"))
    .withColumn("origin_stop_sequence", F.col("origin_stop_sequence").cast("int"))
    .withColumn("destination_stop_sequence", F.col("destination_stop_sequence").cast("int"))
    .withColumn("scheduled_duration_minutes", F.col("scheduled_duration_minutes").cast("double"))
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("is_weekend", F.col("is_weekend").cast("boolean"))
    .withColumn("route_type", F.col("route_type").cast("int")))
print("Loaded rows:", passenger_journeys.count())
passenger_journeys.printSchema()
display(passenger_journeys.limit(5))


Loaded rows: 1000
root
 |-- journey_id: string (nullable = true)
 |-- journey_date: date (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- transport_mode: string (nullable = true)
 |-- origin_stop_id: string (nullable = true)
 |-- origin_stop_name: string (nullable = true)
 |-- origin_stop_sequence: integer (nullable = true)
 |-- destination_stop_id: string (nullable = true)
 |-- destination_stop_name: string (nullable = true)
 |-- destination_stop_sequence: integer (nullable = true)
 |-- scheduled_departure_time: string (nullable = true)
 |-- scheduled_arrival_time: string (nullable = true)
 |-- scheduled_duration_minutes: double (nullable = true)
 |-- passenger_category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- validation_status: string (nullable = true)
 |-- compliance_status: string (nullable = true)
 |-- is_wee

journey_id,journey_date,day_of_week,trip_id,route_id,transport_mode,origin_stop_id,origin_stop_name,origin_stop_sequence,destination_stop_id,destination_stop_name,destination_stop_sequence,scheduled_departure_time,scheduled_arrival_time,scheduled_duration_minutes,passenger_category,payment_method,fare_amount,validation_status,compliance_status,is_weekend,route_type
JNY0000001,2026-10-19,Monday,5935_4928,1 15 c a,Bus,8220DB001159,Bushy Park Hse,15,8220DB001161,Lakelands Pk,17,10:08:30,10:09:42,1.2,Adult,Leap Card,2.54,Validated,Compliant,false,3
JNY0000002,2026-10-02,Friday,5901_1967,1 13 c a,Bus,8230DB002150,Lindisfarne Lawns,16,8230DB001965,Woodford Grove,31,10:14:45,10:32:39,17.9,Adult,Leap Card,2.16,Validated,Compliant,false,3
JNY0000003,2026-10-24,Saturday,5828_27407,3 S8 a,Bus,8230DB002574,De Selby Estate,70,8230DB004929,Verschoyle Avenue,72,13:41:00,13:44:00,3.0,Child,Leap Card,1.07,Validated,Compliant,true,3
JNY0000004,2026-10-08,Thursday,5828_17421,3 L55 a,Bus,8230DB004359,Hollyville Lawn,1,8230DB002202,Kennelsfort Road,4,13:05:00,13:08:00,3.0,Adult,Leap Card,2.57,Validated,Compliant,false,3
JNY0000005,2026-10-03,Saturday,5901_13481,1 44 c a,Bus,8250DB003474,Stepaside Lane,65,8250DB003475,Cairnfort,66,16:41:45,16:42:45,1.0,Adult,Leap Card,2.62,Validated,Compliant,true,3


In [0]:
# Read as strings first: IDs are identifiers, not numeric measures.
raw_csv = (spark.read.option("header", True).option("inferSchema", False)
           .option("mode", "FAILFAST").csv(CSV_PATH))
required = ["journey_id", "journey_date", "day_of_week", "trip_id", "route_id",
            "transport_mode", "origin_stop_id", "origin_stop_name", "origin_stop_sequence",
            "destination_stop_id", "destination_stop_name", "destination_stop_sequence",
            "scheduled_departure_time", "scheduled_arrival_time",
            "scheduled_duration_minutes", "passenger_category", "payment_method",
            "fare_amount", "validation_status", "compliance_status", "is_weekend", "route_type"]
missing = sorted(set(required) - set(raw_csv.columns))
assert not missing, f"Missing columns: {missing}"
assert len(raw_csv.columns) == len(required), "Unexpected CSV columns: inspect source before proceeding."

passenger_journeys = (raw_csv
    .withColumn("journey_date", F.to_date("journey_date", "yyyy-MM-dd"))
    .withColumn("origin_stop_sequence", F.col("origin_stop_sequence").cast("int"))
    .withColumn("destination_stop_sequence", F.col("destination_stop_sequence").cast("int"))
    .withColumn("scheduled_duration_minutes", F.col("scheduled_duration_minutes").cast("double"))
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("is_weekend", F.col("is_weekend").cast("boolean"))
    .withColumn("route_type", F.col("route_type").cast("int")))
print("Loaded rows:", passenger_journeys.count())
passenger_journeys.printSchema()
display(passenger_journeys.limit(5))


Loaded rows: 1000
root
 |-- journey_id: string (nullable = true)
 |-- journey_date: date (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- transport_mode: string (nullable = true)
 |-- origin_stop_id: string (nullable = true)
 |-- origin_stop_name: string (nullable = true)
 |-- origin_stop_sequence: integer (nullable = true)
 |-- destination_stop_id: string (nullable = true)
 |-- destination_stop_name: string (nullable = true)
 |-- destination_stop_sequence: integer (nullable = true)
 |-- scheduled_departure_time: string (nullable = true)
 |-- scheduled_arrival_time: string (nullable = true)
 |-- scheduled_duration_minutes: double (nullable = true)
 |-- passenger_category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- validation_status: string (nullable = true)
 |-- compliance_status: string (nullable = true)
 |-- is_wee

journey_id,journey_date,day_of_week,trip_id,route_id,transport_mode,origin_stop_id,origin_stop_name,origin_stop_sequence,destination_stop_id,destination_stop_name,destination_stop_sequence,scheduled_departure_time,scheduled_arrival_time,scheduled_duration_minutes,passenger_category,payment_method,fare_amount,validation_status,compliance_status,is_weekend,route_type
JNY0000001,2026-10-19,Monday,5935_4928,1 15 c a,Bus,8220DB001159,Bushy Park Hse,15,8220DB001161,Lakelands Pk,17,10:08:30,10:09:42,1.2,Adult,Leap Card,2.54,Validated,Compliant,false,3
JNY0000002,2026-10-02,Friday,5901_1967,1 13 c a,Bus,8230DB002150,Lindisfarne Lawns,16,8230DB001965,Woodford Grove,31,10:14:45,10:32:39,17.9,Adult,Leap Card,2.16,Validated,Compliant,false,3
JNY0000003,2026-10-24,Saturday,5828_27407,3 S8 a,Bus,8230DB002574,De Selby Estate,70,8230DB004929,Verschoyle Avenue,72,13:41:00,13:44:00,3.0,Child,Leap Card,1.07,Validated,Compliant,true,3
JNY0000004,2026-10-08,Thursday,5828_17421,3 L55 a,Bus,8230DB004359,Hollyville Lawn,1,8230DB002202,Kennelsfort Road,4,13:05:00,13:08:00,3.0,Adult,Leap Card,2.57,Validated,Compliant,false,3
JNY0000005,2026-10-03,Saturday,5901_13481,1 44 c a,Bus,8250DB003474,Stepaside Lane,65,8250DB003475,Cairnfort,66,16:41:45,16:42:45,1.0,Adult,Leap Card,2.62,Validated,Compliant,true,3


In [0]:
# Fail closed: do not overwrite existing tables if the replacement file is wrong.
stats = passenger_journeys.agg(
    F.count("*").alias("rows"), F.countDistinct("journey_id").alias("unique_ids"),
    F.min("journey_date").alias("min_date"), F.max("journey_date").alias("max_date"),
    F.sum(F.col("journey_date").isNull().cast("int")).alias("bad_dates"),
    F.sum((F.col("fare_amount").isNull() | (F.col("fare_amount") < 0)).cast("int")).alias("bad_fares"),
    F.sum((F.col("trip_id").isNull() | F.col("route_id").isNull() |
           F.col("journey_id").isNull()).cast("int")).alias("missing_ids"),
    F.sum((F.col("origin_stop_sequence").isNull() | F.col("destination_stop_sequence").isNull() |
           (F.col("origin_stop_sequence") >= F.col("destination_stop_sequence"))).cast("int")).alias("bad_stop_order")
).first()
print(stats.asDict())
assert stats.rows == EXPECTED_ROWS, "Unexpected row count: do not overwrite tables."
assert stats.unique_ids == stats.rows, "Duplicate or null journey IDs."
assert str(stats.min_date) == EXPECTED_START and str(stats.max_date) == EXPECTED_END, "Unexpected date range."
for field in ("bad_dates", "bad_fares", "missing_ids", "bad_stop_order"):
    assert stats[field] == 0, f"Source data quality failure: {field}={stats[field]}"
print("PASS: replacement CSV quality and date checks")


{'rows': 1000, 'unique_ids': 1000, 'min_date': datetime.date(2026, 9, 30), 'max_date': datetime.date(2026, 10, 30), 'bad_dates': 0, 'bad_fares': 0, 'missing_ids': 0, 'bad_stop_order': 0}
PASS: replacement CSV quality and date checks


In [0]:
# Refresh source Delta table only after the replacement CSV passes checks.
(passenger_journeys.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(BRONZE_TABLE))
passenger_journeys = spark.table(BRONZE_TABLE)
print("Refreshed:", BRONZE_TABLE)
print("Row count:", passenger_journeys.count())


Refreshed: workspace.default.passenger_journeys
Row count: 1000


In [0]:
# Create the analytics-ready Silver dataset once (no duplicated rebuild).
journeys_transformed = (passenger_journeys
    .withColumn("journey_year", F.year("journey_date"))
    .withColumn("journey_month", F.month("journey_date"))
    .withColumn("journey_month_name", F.date_format("journey_date", "MMMM"))
    .withColumn("journey_day", F.dayofmonth("journey_date"))
    .withColumn("day_type", F.when(F.dayofweek("journey_date").isin(1, 7), "Weekend").otherwise("Weekday"))
    .withColumn("gtfs_departure_hour", F.split("scheduled_departure_time", ":").getItem(0).cast("int"))
    .withColumn("departure_hour", F.pmod(F.col("gtfs_departure_hour"), F.lit(24)))
    .withColumn("time_period",
        F.when(F.col("departure_hour").between(6, 9), "Morning Peak")
         .when(F.col("departure_hour").between(10, 15), "Daytime")
         .when(F.col("departure_hour").between(16, 19), "Evening Peak")
         .otherwise("Off Peak"))
    .withColumn("is_compliant", F.when(F.col("compliance_status") == "Compliant", 1).otherwise(0))
    .withColumn("is_non_compliant", F.when(F.col("compliance_status") == "Non-Compliant", 1).otherwise(0))
    .withColumn("has_validation_issue", F.when(F.col("compliance_status") == "Validation Issue", 1).otherwise(0))
    .withColumn("fare_band", F.when(F.col("fare_amount") < 2, "Low")
         .when(F.col("fare_amount") < 3, "Medium").otherwise("High")))
print("Transformed:", journeys_transformed.count(), "rows")
display(journeys_transformed.select("journey_id", "journey_date", "trip_id", "transport_mode",
                                    "day_type", "time_period", "fare_band", "compliance_status").limit(10))


Transformed: 1000 rows


journey_id,journey_date,trip_id,transport_mode,day_type,time_period,fare_band,compliance_status
JNY0000001,2026-10-19,5935_4928,Bus,Weekday,Daytime,Medium,Compliant
JNY0000002,2026-10-02,5901_1967,Bus,Weekday,Daytime,Medium,Compliant
JNY0000003,2026-10-24,5828_27407,Bus,Weekend,Daytime,Low,Compliant
JNY0000004,2026-10-08,5828_17421,Bus,Weekday,Daytime,Medium,Compliant
JNY0000005,2026-10-03,5901_13481,Bus,Weekend,Evening Peak,Medium,Compliant
JNY0000006,2026-10-16,5935_6161,Bus,Weekday,Evening Peak,Medium,Compliant
JNY0000007,2026-10-08,5935_14249,Bus,Weekday,Morning Peak,Medium,Compliant
JNY0000008,2026-10-11,5935_11110,Bus,Weekend,Off Peak,High,Compliant
JNY0000009,2026-10-02,5901_110,Bus,Weekday,Daytime,High,Compliant
JNY0000010,2026-10-13,5913_1183,Bus,Weekday,Evening Peak,Medium,Compliant


In [0]:
# Silver checks: no silently lost records or invalid derived fields.
issues = journeys_transformed.agg(
    F.sum((F.col("departure_hour").isNull() | ~F.col("departure_hour").between(0, 23)).cast("int")).alias("invalid_hours"),
    F.sum((F.col("day_type") != F.when(F.col("is_weekend"), "Weekend").otherwise("Weekday")).cast("int")).alias("weekend_mismatch"),
    F.sum(F.col("time_period").isNull().cast("int")).alias("null_periods")
).first()
print("Silver validation:", issues.asDict())
assert journeys_transformed.count() == passenger_journeys.count(), "Source/Silver row mismatch"
assert all(issues[k] == 0 for k in ("invalid_hours", "weekend_mismatch", "null_periods")), "Review derived data"
print("PASS: transformed data quality")


Silver validation: {'invalid_hours': 0, 'weekend_mismatch': 0, 'null_periods': 0}
PASS: transformed data quality


In [0]:
# Publish the refreshed Silver Delta table.
(journeys_transformed.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(SILVER_TABLE))
check = spark.table(SILVER_TABLE)
assert check.count() == EXPECTED_ROWS
print("Saved:", SILVER_TABLE)
display(check.groupBy("journey_date").count().orderBy("journey_date").limit(40))
print("NEXT: Run the updated GTFS notebook's journey/date validations against the refreshed Silver table.")


Saved: workspace.default.passenger_journeys_silver


journey_date,count
2026-09-30,35
2026-10-01,43
2026-10-02,82
2026-10-03,79
2026-10-04,16
2026-10-05,30
2026-10-06,23
2026-10-07,21
2026-10-08,16
2026-10-09,30


NEXT: Run the updated GTFS notebook's journey/date validations against the refreshed Silver table.
